# notebooks/03_complete_forecasting.ipynb
"""
Complete Task 3: Future Forecasting with Confidence Intervals
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TASK 3: FUTURE FORECASTING")
print("="*60)

# Load best model from Task 2
def load_and_forecast(series, order=(2,1,2), forecast_steps=360):
    """Generate forecast with confidence intervals."""
    try:
        # Fit model on full data
        model = ARIMA(series, order=order)
        fitted = model.fit()
        print(f"✓ ARIMA{order} fitted on all data")
        
        # Generate forecast
        forecast = fitted.get_forecast(steps=forecast_steps)
        forecast_mean = forecast.predicted_mean
        forecast_ci = forecast.conf_int(alpha=0.05)
        
        return forecast_mean, forecast_ci, fitted
        
    except Exception as e:
        print(f"Forecast generation failed: {e}")
        raise

# Load data
try:
    df = pd.read_csv('../data/processed/TSLA_data.csv', parse_dates=['Date'])
    df.set_index('Date', inplace=True)
    series = df['Close']
    print(f"Loaded {len(series)} rows")
except Exception as e:
    print(f"Data loading failed: {e}")
    raise

# Generate forecasts
try:
    # 6-month forecast (180 days)
    forecast_6m, ci_6m, model = load_and_forecast(series, order=(2,1,2), forecast_steps=180)
    future_dates_6m = pd.date_range(start=series.index[-1] + pd.Timedelta(days=1), periods=180, freq='B')
    
    # 12-month forecast (360 days)
    forecast_12m, ci_12m, model = load_and_forecast(series, order=(2,1,2), forecast_steps=360)
    future_dates_12m = pd.date_range(start=series.index[-1] + pd.Timedelta(days=1), periods=360, freq='B')
    
    # Calculate forecasted returns
    latest_price = series.iloc[-1]
    forecast_6m_return = (forecast_6m.iloc[-1] / latest_price) - 1
    forecast_12m_return = (forecast_12m.iloc[-1] / latest_price) - 1
    
    print(f"\nLatest Price: ${latest_price:.2f}")
    print(f"\n6-Month Forecast:")
    print(f"  Price: ${forecast_6m.iloc[-1]:.2f}")
    print(f"  Return: {forecast_6m_return*100:.2f}%")
    print(f"  95% CI: [${ci_6m.iloc[-1, 0]:.2f}, ${ci_6m.iloc[-1, 1]:.2f}]")
    
    print(f"\n12-Month Forecast:")
    print(f"  Price: ${forecast_12m.iloc[-1]:.2f}")
    print(f"  Return: {forecast_12m_return*100:.2f}%")
    print(f"  95% CI: [${ci_12m.iloc[-1, 0]:.2f}, ${ci_12m.iloc[-1, 1]:.2f}]")
    
except Exception as e:
    print(f"Forecast failed: {e}")
    raise

# Plot with clear distinction
fig, ax = plt.subplots(figsize=(16, 8))

# Historical data (last 500 days)
historical = series.iloc[-500:]
ax.plot(historical.index, historical, label='Historical Data', color='blue', linewidth=2)

# Future forecast - 12 months
ax.plot(future_dates_12m, forecast_12m, 
        label='12-Month Forecast', color='red', linewidth=2, linestyle='--')

# Confidence intervals
ax.fill_between(future_dates_12m, 
                 ci_12m.iloc[:, 0], 
                 ci_12m.iloc[:, 1], 
                 color='red', alpha=0.15, label='95% Confidence Interval')

# Vertical line at forecast start
ax.axvline(x=series.index[-1], color='black', linestyle='-', alpha=0.5, linewidth=1)
ax.text(series.index[-1], ax.get_ylim()[1]*0.95, 'Forecast Start', 
        ha='center', va='top', fontsize=10, rotation=90)

ax.set_title('TSLA: 12-Month Future Forecast with Confidence Intervals', 
             fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("TASK 3 COMPLETE")
print("="*60)